# LFW Grad-CAM — 00. Source, scope, alignment, and model freeze

압축 결과가 아니라 **선택 표본, 공통 112×112 정렬 crop, ModelSpec**을
먼저 동결합니다. 이 단계에서 정량 결과나 사례 목록을 요구하지 않습니다.

`DATA_FRACTION`은 split별 identity 전체를 결정적으로 선택합니다. 행 단위
절단은 동일인 leave-one-out 구성을 깨뜨리므로 사용하지 않습니다.


In [1]:
# cell 1 : 실행 모드 설정 및 조건 검증
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_PROFILE = "arcface_ms1mv3_r100"     # arcface, adaface, magface 중 이번 실행 profile
MODE = "dev"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = True      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = True      # 새 immutable artifact 저장 시에만 True

all_profiles = CONFIG["models"]["selected_profiles"] + CONFIG["models"].get("bridge_profiles", [])
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")
if MODEL_PROFILE not in available_profiles:
    raise ValueError(f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}")
PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [2]:
# cell 2 : 입력 및 출력 경로 지정
import hashlib
import json

import numpy as np
import pandas as pd

from research.embeddings import select_model_spec_by_profile
from research.experiments.scope import (
    ExperimentScope,
    select_manifest_fraction,
)
from research.runtime.hashing import sha256_file

MODEL_REGISTRY_ROOT = PROJECT_ROOT / "runs/step2/model_registry"
MODEL_UID = None  # 같은 family가 2개 이상 등록된 경우 exact UID 지정
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / CONFIG["datasets"]["lfw"]["manifest_path"]
)

# YAML config에서 aligned crop 경로를 읽음
LFW_ALIGNED = CONFIG["datasets"]["lfw"]["aligned_crops"]
ALIGNED_CROP_MANIFEST_PATH = PROJECT_ROOT / LFW_ALIGNED["manifest_path"]
ALIGNED_FACES_NPY_PATH = PROJECT_ROOT / LFW_ALIGNED["array_path"]

SELECTED_MANIFEST_OUTPUT_PATH = None  # 지정하지 않으면 model_uid/extraction_uid로 자동 설정
FREEZE_MANIFEST_OUTPUT_PATH = None    # 지정하지 않으면 model_uid/extraction_uid로 자동 설정
REGION_MASK_BUNDLE_PATH = None  # dense-landmark/face mask가 있을 때만


In [3]:
# cell 3 : 선택 표본·모델·crop 동결 실행
def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


if EXECUTE_STAGE:
    required_paths = {
        "source_manifest": SOURCE_MANIFEST_PATH,
        "aligned_crop_manifest": ALIGNED_CROP_MANIFEST_PATH,
        "aligned_faces_npy": ALIGNED_FACES_NPY_PATH,
    }
    missing = [name for name, value in required_paths.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    resolved = {
        name: Path(value).resolve()
        for name, value in required_paths.items()
    }
    for name, path in resolved.items():
        if not path.is_file():
            raise FileNotFoundError(f"{name}: {path}")

    model_spec_path, spec = select_model_spec_by_profile(
        MODEL_REGISTRY_ROOT,
        profile_id=MODEL_PROFILE,
        profile_config=PROFILE,
        verify_checkpoint=True,
    )
    resolved["model_spec"] = model_spec_path

    aligned = read_table(resolved["aligned_crop_manifest"]).copy()
    if "image_id" in aligned and "sample_id" not in aligned:
        aligned = aligned.rename(columns={"image_id": "sample_id"})
    aligned_required = {
        "sample_id",
        "aligned_face_index",
        "aligned_content_sha256",
    }
    if missing_columns := sorted(aligned_required.difference(aligned.columns)):
        raise ValueError(f"aligned manifest 누락 열: {missing_columns}")
    if aligned["sample_id"].astype(str).duplicated().any():
        raise ValueError("aligned manifest sample_id가 중복됩니다.")

    source = read_table(resolved["source_manifest"]).copy()
    if "image_id" in source and "sample_id" not in source:
        source = source.rename(columns={"image_id": "sample_id"})
    required = {"sample_id", "identity_id", "split"}
    if missing_columns := sorted(required.difference(source.columns)):
        raise ValueError(f"source manifest 누락 열: {missing_columns}")

    source["sample_id"] = source["sample_id"].astype(str)
    aligned["sample_id"] = aligned["sample_id"].astype(str)
    valid_sample_ids = set(aligned["sample_id"])
    source = source[source["sample_id"].isin(valid_sample_ids)].reset_index(drop=True)

    scope = ExperimentScope(
        mode=MODE,
        data_fraction=DATA_FRACTION,
        seed=SEED,
    )
    selected_parts = [
        select_manifest_fraction(
            rows,
            scope,
            namespace=f"step2:lfw:{split_name}",
        )
        for split_name, rows in source.groupby("split", sort=True)
    ]
    selected = pd.concat(selected_parts, ignore_index=True)
    selected = selected.merge(
        aligned.loc[:, sorted(aligned_required)],
        on="sample_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    if (selected["_merge"] != "both").any():
        raise ValueError("선택 표본 중 공통 정렬 crop이 없는 행이 있습니다.")
    selected = (
        selected.drop(columns="_merge")
        .sort_values("aligned_face_index", kind="stable")
        .reset_index(drop=True)
    )
    indices = selected["aligned_face_index"].to_numpy(dtype=np.int64)
    if len(set(indices.tolist())) != len(indices) or np.any(indices < 0):
        raise ValueError("aligned_face_index는 고유한 음이 아닌 정수여야 합니다.")
    aligned_faces = np.load(
        resolved["aligned_faces_npy"],
        mmap_mode="r",
        allow_pickle=False,
    )
    if (
        aligned_faces.ndim != 4
        or aligned_faces.shape[1:] != (112, 112, 3)
        or aligned_faces.dtype != np.uint8
        or int(indices.max()) >= len(aligned_faces)
    ):
        raise ValueError("aligned face 배열 계약이 맞지 않습니다.")

    selection_bytes = selected.to_csv(index=False).encode("utf-8")
    selection_uid = hashlib.sha256(selection_bytes).hexdigest()
    selected["template_scope_id"] = (
        selected["split"].astype(str) + ":" + selection_uid[:16]
    )
    selected_content_sha256 = hashlib.sha256(
        selected.to_csv(index=False).encode("utf-8")
    ).hexdigest()
    extraction_payload = "\x1f".join(
        [
            "lfw",
            spec.model_uid,
            selected_content_sha256,
            sha256_file(CONFIG_PATH),
            MODE,
            str(DATA_FRACTION),
            str(SEED),
        ]
    )
    extraction_uid = (
        "population-"
        + hashlib.sha256(extraction_payload.encode("utf-8")).hexdigest()[:24]
    )

    # 자동 출력 경로: model_uid/extraction_uid 기반 (명시적 지정 시 우회)
    run_dir = (
        PROJECT_ROOT
        / CONFIG["run"]["root"]
        / "lfw"
        / spec.model_uid
        / extraction_uid
    )
    selected_path = (
        Path(SELECTED_MANIFEST_OUTPUT_PATH).resolve()
        if SELECTED_MANIFEST_OUTPUT_PATH is not None
        else run_dir / "selected_manifest.parquet"
    )
    freeze_path = (
        Path(FREEZE_MANIFEST_OUTPUT_PATH).resolve()
        if FREEZE_MANIFEST_OUTPUT_PATH is not None
        else run_dir / "freeze_manifest.json"
    )

    freeze_manifest = {
        "schema_version": 1,
        "dataset_id": "lfw",
        "model_uid": spec.model_uid,
        "checkpoint_sha256": spec.checkpoint.sha256,
        "preprocess_hash": spec.preprocessing.preprocess_hash,
        "target_layer": spec.target_layer,
        "extraction_uid": extraction_uid,
        "scope": scope.as_dict(),
        "selected_sample_count": int(len(selected)),
        "selected_identity_count": int(selected["identity_id"].nunique()),
        "selected_manifest_content_sha256": selected_content_sha256,
        "inputs": {
            name: {"path": str(path), "sha256": sha256_file(path)}
            for name, path in resolved.items()
        },
        "region_mask_bundle": (
            None
            if REGION_MASK_BUNDLE_PATH is None
            else {
                "path": str(Path(REGION_MASK_BUNDLE_PATH).resolve()),
                "sha256": sha256_file(
                    Path(REGION_MASK_BUNDLE_PATH).resolve()
                ),
            }
        ),
        "fallback_free": True,
    }
    if WRITE_OUTPUTS:
        for path in (selected_path, freeze_path):
            if path.exists():
                raise FileExistsError(f"기존 artifact를 덮어쓸 수 없습니다: {path}")
            path.parent.mkdir(parents=True, exist_ok=True)
        selected.to_parquet(selected_path, index=False)
        freeze_manifest["selected_manifest_file"] = {
            "path": str(selected_path),
            "sha256": sha256_file(selected_path),
        }
        freeze_path.write_text(
            json.dumps(freeze_manifest, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
else:
    freeze_manifest = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
freeze_manifest


{'schema_version': 1,
 'dataset_id': 'lfw',
 'model_uid': 'arcface-7972a704552df378345f',
 'checkpoint_sha256': 'a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b',
 'preprocess_hash': '3976664d86e41199f7fd522aa9c1d70decf380ca8b2da9b6d2ee84d6dc4c03a9',
 'target_layer': 'layer4.2.conv2',
 'extraction_uid': 'population-c1b696495d2787e584b4639d',
 'scope': {'mode': 'dev',
  'data_fraction': 1.0,
  'seed': 42,
  'is_full_dataset': True,
  'is_paper_run': False},
 'selected_sample_count': 13195,
 'selected_identity_count': 5736,
 'selected_manifest_content_sha256': '63a1a68b48767833acda4af092c70757f030dd4e3bb906ead6c39b7ecab68379',
 'inputs': {'source_manifest': {'path': 'C:\\ronbun\\data\\interim\\lfw\\face_manifest.csv',
   'sha256': '839d228a63995cd456fa2045c29702c8271cbbaaeb99a033bce767245657ea2c'},
  'aligned_crop_manifest': {'path': 'C:\\ronbun\\data\\interim\\common\\aligned_112\\lfw\\aligned_manifest.parquet',
   'sha256': 'b75e32ae32a14c71c52b17c21b09d9b3891585def083

hash, 선택 범위 또는 정렬 index가 달라지면 기존 run을 이어 쓰지 않고
00부터 새 lineage로 시작합니다.


In [4]:
# 누락된 표본 확인
missing_samples = selected[selected["_merge"] == "left_only"]
print(f"누락된 표본 개수: {len(missing_samples)}")
print(missing_samples[["sample_id", "identity_id", "split"]])


KeyError: '_merge'